<a href="https://colab.research.google.com/github/sumantabhargab/SAFE5G/blob/main/02_train_model_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("="*60)
print("SAFE5G v2")
print("Notebook 2 : Model Training")
print("="*60)

SAFE5G v2
Notebook 2 : Model Training


In [ ]:
!pip install timm torchmetrics

In [ ]:
import torch
import timm
import numpy as np
import matplotlib.pyplot as plt

from torchvision import datasets,transforms
from torch.utils.data import DataLoader

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    f1_score,
    precision_score,
    recall_score
)

print("Libraries Loaded")

Libraries Loaded


In [ ]:
import torch
import timm
import numpy as np
import matplotlib.pyplot as plt

from torchvision import datasets,transforms
from torch.utils.data import DataLoader

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    f1_score,
    precision_score,
    recall_score
)

print("Libraries Loaded")

Libraries Loaded


In [ ]:
ROOT = "/content/Safe5G_Frames"

In [ ]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

IMAGE_SIZE = 224
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05,0.05),
        scale=(0.95,1.05)
    ),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [ ]:
import os

print(os.path.exists("/content/Safe5G_Frames"))
print("Testing Images :", len(test_dataset))

False


NameError: name 'test_dataset' is not defined

In [ ]:
import torch

print("="*50)
print("PyTorch Version :", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print("GPU Count :", torch.cuda.device_count())

print("="*50)

PyTorch Version : 2.11.0+cu128
CUDA Available : True
GPU : Tesla T4
GPU Count : 1


In [ ]:
!pip install -q timm torchinfo torchmetrics tensorboard

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 25.0 MB/s eta 0:00:00


In [ ]:
import os
import time
import copy
import json
import random
import warnings
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import timm

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve
)

from torch.cuda.amp import autocast, GradScaler

print("Libraries Loaded.")

Libraries Loaded.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
##############################################
# CONFIG
##############################################

ROOT = "/content/drive/MyDrive/Safe5G/Safe5G_Frames"

MODEL_SAVE_PATH = "/content/drive/MyDrive/Safe5G/Models"

RESULT_PATH = "/content/drive/MyDrive/Safe5G/Results"

MODEL_NAME = "efficientnet_b0"

IMAGE_SIZE = 224

BATCH_SIZE = 32

EPOCHS = 25

LEARNING_RATE = 1e-4

WEIGHT_DECAY = 1e-4

EARLY_STOPPING = 5

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device :", DEVICE)

if torch.cuda.is_available():

    print(torch.cuda.get_device_name(0))

os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
os.makedirs(RESULT_PATH, exist_ok=True)

Device : cuda
Tesla T4


In [ ]:
train_transform = transforms.Compose([

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(10),

    transforms.RandomAffine(

        degrees=0,

        translate=(0.05,0.05),

        scale=(0.95,1.05)

    ),

    transforms.ColorJitter(

        brightness=0.15,

        contrast=0.15,

        saturation=0.15

    ),

    transforms.ToTensor(),

    transforms.Normalize(

        [0.485,0.456,0.406],

        [0.229,0.224,0.225]

    )

])

test_transform = transforms.Compose([

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(

        [0.485,0.456,0.406],

        [0.229,0.224,0.225]

    )

])

In [ ]:
train_dataset = datasets.ImageFolder(

    ROOT + "/train",

    transform=train_transform

)

val_dataset = datasets.ImageFolder(

    ROOT + "/val",

    transform=test_transform

)

test_dataset = datasets.ImageFolder(

    ROOT + "/test",

    transform=test_transform

)

print(train_dataset.class_to_idx)

print()

print("Train :",len(train_dataset))

print("Validation :",len(val_dataset))

print("Test :",len(test_dataset))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Safe5G/train'

In [ ]:
fig = plt.figure(figsize=(10,10))

for i in range(16):

    image,label = train_dataset[random.randint(0,len(train_dataset)-1)]

    image = image.permute(1,2,0)

    image = image.numpy()

    image = image * np.array([0.229,0.224,0.225])

    image = image + np.array([0.485,0.456,0.406])

    image = np.clip(image,0,1)

    plt.subplot(4,4,i+1)

    plt.imshow(image)

    plt.title(label)

    plt.axis("off")

plt.show()

In [ ]:
train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=2,

    pin_memory=True

)

val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=2,

    pin_memory=True

)

test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=2,

    pin_memory=True

)

print("Train Batches :",len(train_loader))

print("Validation Batches :",len(val_loader))

print("Test Batches :",len(test_loader))

In [ ]:
model = timm.create_model(

    MODEL_NAME,

    pretrained=True,

    num_classes=2

)

model = model.to(DEVICE)

print(model)

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

EfficientNet(
  (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNormAct2d(
    32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
    (drop): Identity()
    (act): SiLU(inplace=True)
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): DepthwiseSeparableConv(
        (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (bn1): BatchNormAct2d(
          32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): SiLU(inplace=True)
        )
        (aa): Identity()
        (se): SqueezeExcite(
          (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
          (act1): SiLU(inplace=True)
          (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
          (gate): Sigmoid()
        )
        (conv_pw): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn2

In [ ]:
import os

ROOT = "/content/drive/MyDrive/Safe5G"

for root, dirs, files in os.walk(ROOT):
    print(root)

/content/drive/MyDrive/Safe5G
/content/drive/MyDrive/Safe5G/Safe5G_Frames
/content/drive/MyDrive/Safe5G/Safe5G_Frames/val
/content/drive/MyDrive/Safe5G/Safe5G_Frames/val/Violence
/content/drive/MyDrive/Safe5G/Safe5G_Frames/val/NonViolence
/content/drive/MyDrive/Safe5G/Safe5G_Frames/train
/content/drive/MyDrive/Safe5G/Safe5G_Frames/train/NonViolence
/content/drive/MyDrive/Safe5G/Safe5G_Frames/train/Violence
/content/drive/MyDrive/Safe5G/Safe5G_Dataset
/content/drive/MyDrive/Safe5G/Safe5G_Dataset/val
/content/drive/MyDrive/Safe5G/Safe5G_Dataset/val/Violence
/content/drive/MyDrive/Safe5G/Safe5G_Dataset/val/NonViolence
/content/drive/MyDrive/Safe5G/Models
/content/drive/MyDrive/Safe5G/Results
/content/drive/MyDrive/Safe5G/Notebooks


In [ ]:
print(os.listdir("/content/drive/MyDrive/Safe5G/Safe5G_Frames"))

['val', 'train', 'test']


In [ ]:
print(os.listdir("/content/Safe5G_Frames/test"))

FileNotFoundError: [Errno 2] No such file or directory: '/content/Safe5G_Frames/test'

In [ ]:
print(os.listdir("/content/drive/MyDrive/Safe5G/Safe5G_Frames/test"))

['Violence', 'NonViolence']


In [ ]:
import os

ROOT = "/content/drive/MyDrive/Safe5G/Safe5G_Frames"

for split in ["train", "val", "test"]:
    print(f"\n{split.upper()}")

    split_path = os.path.join(ROOT, split)

    print("Exists:", os.path.exists(split_path))

    if os.path.exists(split_path):
        print(os.listdir(split_path))


TRAIN
Exists: True
['NonViolence', 'Violence']

VAL
Exists: True
['Violence', 'NonViolence']

TEST
Exists: True
['Violence', 'NonViolence']


In [ ]:
from torchvision.datasets import ImageFolder

dataset = ImageFolder("/content/drive/MyDrive/Safe5G/Safe5G_Frames/train")

print(dataset.class_to_idx)
print(len(dataset))

{'NonViolence': 0, 'Violence': 1}
16575
